In [1]:
! pip install soundfile


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [2]:
! pip install datasets


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [3]:
! pip install scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [4]:
! pip install tensorflow


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [5]:
! pip install numpy pandas


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [6]:
! pip install librosa


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [7]:
! pip install keras


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [8]:
! pip install torch


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [9]:
! pip install torchcodec


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [10]:
import os
import re
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import tensorflow as tf
from datasets import load_dataset

/Users/varshh06/DL-END-TO-END/DL-END-TO-END/.venv-2/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
import sys
print(sys.executable)
print(sys.version)

/Users/varshh06/DL-END-TO-END/DL-END-TO-END/.venv-2/bin/python
3.14.2 (v3.14.2:df793163d58, Dec  5 2025, 12:18:06) [Clang 16.0.0 (clang-1600.0.26.6)]


In [12]:
import torch
import torchcodec
import datasets

print("PyTorch:", torch.__version__)
print("TorchCodec:", torchcodec.__file__)
print("Datasets:", datasets.__version__)

PyTorch: 2.14.1
TorchCodec: /Users/varshh06/DL-END-TO-END/DL-END-TO-END/.venv-2/lib/python3.14/site-packages/torchcodec/__init__.py
Datasets: 5.1.0


In [13]:
from tensorflow.keras import layers, Model

In [14]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)

TensorFlow: 2.22.0-rc0


In [15]:
ARTIFACT_DIR = "artifacts"
DATA_DIR = "librispeech_audio"
TRAIN_LIMIT = 1200       # raise this for more training data; None means stream the whole split
EVAL_LIMIT = 200         # per validation/test split; raise for more reliable evaluation
EPOCHS = 25
BATCH_SIZE = 8
LEARNING_RATE = 3e-4

In [16]:
SAMPLE_RATE = 16000
N_MFCC = 40
MAX_SECONDS = 12
MIN_SECONDS = 0.15
MAX_FRAMES = int(MAX_SECONDS * SAMPLE_RATE / 160) + 2

In [17]:
os.makedirs(ARTIFACT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

In [18]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z' ]", "", text)
    return re.sub(r"\s+", " ", text).strip()

In [19]:
def extract_features(path):
    try:
        audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True, duration=MAX_SECONDS)
        if len(audio) < int(MIN_SECONDS * SAMPLE_RATE):
            return None
        features = librosa.feature.mfcc(
            y=audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
            n_fft=400, hop_length=160
        ).T.astype(np.float32)
        mean = features.mean(axis=0, keepdims=True)
        std = features.std(axis=0, keepdims=True)
        features = (features - mean) / (std + 1e-5)
        return features[:MAX_FRAMES]
    except Exception as exc:
        print(f"Skipping unreadable audio {path}: {exc}")
        return None

In [20]:
def minimum_ctc_frames(label_ids):
    repeats = sum(label_ids[i] == label_ids[i - 1] for i in range(1, len(label_ids)))
    return len(label_ids) + repeats

In [21]:
from sklearn.model_selection import train_test_split


In [ ]:
import sys
import traceback

print("Python:", sys.version)
print("Environment:", sys.executable)

try:
    from torchcodec.decoders import AudioDecoder
    print(" TorchCodec AudioDecoder is working!")
except Exception:
    print(" TorchCodec audio decoder failed:")
    traceback.print_exc()

Python: 3.14.2 (v3.14.2:df793163d58, Dec  5 2025, 12:18:06) [Clang 16.0.0 (clang-1600.0.26.6)]
Environment: /Users/varshh06/DL-END-TO-END/DL-END-TO-END/.venv-2/bin/python
✅ TorchCodec AudioDecoder is working!


In [29]:
import os
import shutil

print("FFmpeg path:", shutil.which("ffmpeg"))
print("PATH:", os.environ.get("PATH"))

FFmpeg path: None
PATH: /Users/varshh06/DL-END-TO-END/DL-END-TO-END/.venv-2/bin:/usr/bin:/bin:/usr/sbin:/sbin:/usr/local/bin


In [30]:
import os
import shutil

os.environ["PATH"] = "/opt/homebrew/bin:" + os.environ.get("PATH", "")

print("FFmpeg path:", shutil.which("ffmpeg"))

FFmpeg path: /opt/homebrew/bin/ffmpeg


In [32]:
def download_split(split_name, limit, prefix):
    """Stream LibriSpeech examples and save audio locally as WAV files."""
    print(f"Loading dataset split: {split_name}")

    ds = load_dataset(
        "openslr/librispeech_asr",
        "clean",
        split=split_name,
        streaming=True
    )

    records = []

    for index, item in enumerate(ds):
        if limit is not None and index >= limit:
            break

        text = clean_text(item.get("text", ""))
        audio_info = item.get("audio")

        if not text or audio_info is None:
            continue

        # Decode audio, supporting both old and new Hugging Face formats
        try:
            if isinstance(audio_info, dict):
                audio_array = np.asarray(
                    audio_info["array"], dtype=np.float32
                )
                sampling_rate = int(
                    audio_info.get("sampling_rate", SAMPLE_RATE)
                )
            else:
                audio_data = audio_info.get_all_samples()

                audio_array = (
                    audio_data.data
                    .squeeze(0)
                    .cpu()
                    .numpy()
                    .astype(np.float32)
                )
                sampling_rate = int(audio_data.sample_rate)

            # Ensure the waveform is one-dimensional
            audio_array = np.asarray(audio_array, dtype=np.float32).squeeze()

            if audio_array.ndim != 1 or audio_array.size == 0:
                print(f"  Skipping invalid audio at index {index}")
                continue

            # Resample to the target sample rate
            if sampling_rate != SAMPLE_RATE:
                audio_array = librosa.resample(
                    audio_array,
                    orig_sr=sampling_rate,
                    target_sr=SAMPLE_RATE
                )

            # Skip clips that are too short
            if len(audio_array) < int(MIN_SECONDS * SAMPLE_RATE):
                continue

            audio_path = os.path.join(
                DATA_DIR, f"{prefix}_{index:06d}.wav"
            )

            # Save audio if it doesn't already exist
            if not os.path.isfile(audio_path):
                sf.write(audio_path, audio_array, SAMPLE_RATE)

            records.append((audio_path, text))

            if len(records) % 100 == 0:
                print(f"  Saved {len(records)} usable clips")

        except Exception as e:
            print(f"  Skipping index {index} due to audio error: {e}")
            continue

    print(f"Finished {split_name}: {len(records)} clips")
    return records


# Download the training, validation, and test splits
train_records = download_split("train.100", TRAIN_LIMIT, "train")
val_records = download_split("validation", EVAL_LIMIT, "validation")
test_records = download_split("test", EVAL_LIMIT, "test")


# Verify that all splits contain usable audio
if min(
    len(train_records),
    len(val_records),
    len(test_records)
) == 0:
    raise RuntimeError(
        "A dataset split is empty. Check internet access, "
        "the Hugging Face dataset name, and the split names."
    )

print(
    f"Records — train: {len(train_records)}, "
    f"validation: {len(val_records)}, "
    f"test: {len(test_records)}"
)


# Build the character-level vocabulary for CTC
CHARACTERS = list(" abcdefghijklmnopqrstuvwxyz'")

char_to_id = {
    char: i + 1
    for i, char in enumerate(CHARACTERS)
}

id_to_char = {
    i: char
    for char, i in char_to_id.items()
}

NUM_CHARACTERS = len(CHARACTERS)
BLANK_ID = 0


# Save the vocabulary
vocab_path = os.path.join(ARTIFACT_DIR, "vocab.json")

with open(vocab_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "char_to_id": char_to_id,
            "id_to_char": {
                str(k): v for k, v in id_to_char.items()
            },
            "blank_id": BLANK_ID
        },
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"Vocabulary saved to: {vocab_path}")
print(f"Number of characters: {NUM_CHARACTERS}")
print(f"CTC blank ID: {BLANK_ID}")

Loading dataset split: train.100
  Saved 100 usable clips
  Saved 200 usable clips
  Saved 300 usable clips
  Saved 400 usable clips
  Saved 500 usable clips
  Saved 600 usable clips
  Saved 700 usable clips
  Saved 800 usable clips
  Saved 900 usable clips
  Saved 1000 usable clips
  Saved 1100 usable clips
  Saved 1200 usable clips
Finished train.100: 1200 clips
Loading dataset split: validation
  Saved 100 usable clips
  Saved 200 usable clips
Finished validation: 200 clips
Loading dataset split: test
  Saved 100 usable clips
  Saved 200 usable clips
Finished test: 200 clips
Records — train: 1200, validation: 200, test: 200
Vocabulary saved to: artifacts/vocab.json
Number of characters: 28
CTC blank ID: 0


In [ ]:
def prepare_records(raw_records):
    prepared = []
    skipped = 0
    for path, text in raw_records:
        ids = [char_to_id[c] for c in text if c in char_to_id]
        if not ids:
            skipped += 1
            continue
        features = extract_features(path)
        if features is None or len(features) < minimum_ctc_frames(ids):
            skipped += 1
            continue
        prepared.append({"path": path, "text": text, "features": features, "labels": np.asarray(ids, dtype=np.int32)})
    print(f"Prepared {len(prepared)} / {len(raw_records)}; skipped {skipped}")
    return prepared

train_data = prepare_records(train_records)
val_data = prepare_records(val_records)
test_data = prepare_records(test_records)
if not train_data or not val_data or not test_data:
    raise ValueError("A split is empty after preprocessing. Increase dataset limits or inspect the downloaded data.")


In [ ]:
class SpeechSequence(tf.keras.utils.Sequence):
    def __init__(self, samples, batch_size=8, shuffle=False):
        super().__init__()
        self.samples = samples
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(samples))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.samples) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, index):
        batch_ids = self.indices[index * self.batch_size:(index + 1) * self.batch_size]
        batch = [self.samples[i] for i in batch_ids]
        max_t = max(len(item["features"]) for item in batch)
        max_l = max(len(item["labels"]) for item in batch)
        audio = np.zeros((len(batch), max_t, N_MFCC), dtype=np.float32)
        labels = np.zeros((len(batch), max_l), dtype=np.int32)
        input_length = np.zeros((len(batch), 1), dtype=np.int32)
        label_length = np.zeros((len(batch), 1), dtype=np.int32)
        for i, item in enumerate(batch):
            f, y = item["features"], item["labels"]
            audio[i, :len(f)] = f
            labels[i, :len(y)] = y
            input_length[i, 0] = len(f)
            label_length[i, 0] = len(y)
        inputs = {"audio": audio, "labels": labels, "input_length": input_length, "label_length": label_length}
        return inputs, np.zeros((len(batch), 1), dtype=np.float32)

In [ ]:
train_seq = SpeechSequence(train_data, BATCH_SIZE, shuffle=True)
val_seq = SpeechSequence(val_data, BATCH_SIZE, shuffle=False)
test_seq = SpeechSequence(test_data, BATCH_SIZE, shuffle=False)

In [ ]:
def make_model(num_characters):
    audio = layers.Input(shape=(None, N_MFCC), dtype=tf.float32, name="audio")
    labels = layers.Input(shape=(None,), dtype=tf.int32, name="labels")
    input_length = layers.Input(shape=(1,), dtype=tf.int32, name="input_length")
    label_length = layers.Input(shape=(1,), dtype=tf.int32, name="label_length")

    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(audio)
    x = layers.Dropout(0.25)(x)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
    x = layers.Dropout(0.25)(x)
    probabilities = layers.Dense(num_characters + 1, activation="softmax", name="character_probabilities")(x)

    def ctc_loss(args):
        y_true, y_pred, in_len, lab_len = args
        return tf.keras.backend.ctc_batch_cost(y_true, y_pred, in_len, lab_len)

    loss_output = layers.Lambda(ctc_loss, name="ctc_loss")([labels, probabilities, input_length, label_length])
    training_model = Model([audio, labels, input_length, label_length], loss_output)
    training_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE, clipnorm=1.0),
        loss=lambda _y_true, y_pred: y_pred
    )
    inference_model = Model(audio, probabilities)
    return training_model, inference_model

training_model, inference_model = make_model(NUM_CHARACTERS)
training_model.summary()

In [ ]:
def decode_probabilities(probabilities, input_length):
    decoded, _ = tf.keras.backend.ctc_decode(
        probabilities,
        input_length=np.asarray([input_length], dtype=np.int32),
        greedy=True,
        beam_width=1
    )
    ids = decoded[0].numpy()[0]
    return "".join(id_to_char.get(int(i), "") for i in ids if int(i) > 0).strip()

In [ ]:
def edit_distance(reference, hypothesis):
    previous = list(range(len(hypothesis) + 1))
    for i, ref_char in enumerate(reference, start=1):
        current = [i]
        for j, hyp_char in enumerate(hypothesis, start=1):
            current.append(min(
                current[j - 1] + 1,
                previous[j] + 1,
                previous[j - 1] + (ref_char != hyp_char)
            ))
        previous = current
    return previous[-1]

In [ ]:

def corpus_error_rates(samples, model):
    word_errors = word_count = char_errors = char_count = 0
    examples = []
    for item in samples:
        features = item["features"]
        probs = model.predict(features[None, ...], verbose=0)
        prediction = decode_probabilities(probs, len(features))
        reference = item["text"].strip()
        ref_words, hyp_words = reference.split(), prediction.split()
        word_errors += edit_distance(ref_words, hyp_words)
        word_count += max(1, len(ref_words))
        char_errors += edit_distance(reference, prediction)
        char_count += max(1, len(reference))
        if len(examples) < 5:
            examples.append((reference, prediction))
    return word_errors / word_count, char_errors / char_count, examples

In [ ]:
class RecognitionMetrics(tf.keras.callbacks.Callback):
    def __init__(self, validation_samples):
        super().__init__()
        self.validation_samples = validation_samples

    def on_epoch_end(self, epoch, logs=None):
        wer, cer, _ = corpus_error_rates(self.validation_samples, inference_model)
        logs = logs or {}
        logs["val_wer"] = wer
        logs["val_cer"] = cer
        print(f" — val_WER: {wer:.4f} — val_CER: {cer:.4f}")

In [ ]:
best_model_path = os.path.join(ARTIFACT_DIR, "best_speech_model.weights.h5")
metrics_callback = RecognitionMetrics(val_data)
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(best_model_path, monitor="val_loss", save_best_only=True, save_weights_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    metrics_callback
]

In [ ]:
history = training_model.fit(
    train_seq,
    validation_data=val_seq,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

In [ ]:

if os.path.exists(best_model_path):
    training_model.load_weights(best_model_path)

inference_model.save(os.path.join(ARTIFACT_DIR, "speech_model.keras"))
print("Saved best checkpoint, inference model, and vocabulary in", ARTIFACT_DIR)

In [ ]:
test_wer, test_cer, examples = corpus_error_rates(test_data, inference_model)
print(f"Test WER: {test_wer:.4f} ({test_wer * 100:.2f}%)")
print(f"Test CER: {test_cer:.4f} ({test_cer * 100:.2f}%)")
print("\nSample predictions:")
for reference, prediction in examples:
    print("REFERENCE :", reference)
    print("PREDICTION:", prediction)
    print()

In [ ]:
def transcribe(audio_path):
    features = extract_features(audio_path)
    if features is None:
        raise ValueError("Audio is too short or could not be read.")
    probabilities = inference_model.predict(features[None, ...], verbose=0)
    return decode_probabilities(probabilities, len(features))

# Example after training:
# print(transcribe("/path/to/your/audio.wav"))